# Download, filter, and visualize conversational selected activations

This notebook works with the batch files produced by `scripts/run_activation_caching_conversational_selected_acts.sh`. It downloads the self-contained activation batches from the hardcoded GCS path, filters rows using their embedded prompt metadata, loads `layer_out/21` at the cached final-token position, optionally averages matching metadata groups, projects the activations with PCA, and visualizes the result.

## 1. Setup

For a fresh Colab runtime, uncomment the clone and authentication commands. Local runs can skip them.

In [ ]:
# !git clone -b dev https://github.com/justinshenk/temporal-manifolds.git
%cd temporal-manifolds
# !gcloud auth application-default login
# !mv -n .env.example .env

In [ ]:
import gc
import os
import pickle
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from google.cloud import storage
from sklearn.decomposition import PCA
from tqdm.auto import tqdm

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
load_dotenv(repo_root / '.env')

## 2. Configure the workflow

`METADATA_FILTERS` accepts any embedded prompt-metadata field, with dotted paths for nested fields. A scalar requires an exact match; a list keeps any listed value. Different fields are combined with AND. Use `None` or `{}` to disable filtering. `DIRECTIONS` accepts a list of one-dimensional tensors, NumPy arrays, or array-like vectors. Each direction is normalized, projected out of every activation vector, and applied sequentially in list order before aggregation and PCA. Use `[]` to disable direction removal. `AGG_BY` lists flattened metadata fields whose equal values define averaging groups; the derived `time_horizon_months` field is supported. Use `None` or `[]` to retain individual rows.

In [ ]:
METADATA_FILTERS: dict[str, object] | None = {
    'template_metadata.prompt_framing': 'task_available_time',
    # 'task_metadata.domain': 'communication',
    # 'template_metadata.output_format': 'approach_actions',
}

# Output contract of scripts/run_activation_caching_conversational_selected_acts.sh.
ACTIVATIONS_GCS_URI = 'gs://temporal-research-bucket/selected_acts'
DATA_DIR = repo_root / 'data' / 'filtered_conversational_selected_activations'
PROJECT_ID = os.getenv('GCP_PROJECT_ID')
OVERWRITE = False
DOWNLOAD_WORKERS = 8
MAX_SAMPLES: int | None = None
SELECTED_BATCH_FILES: list[str] | None = None

LAYER_COMPONENT = 'layer_out/21'
POSITION_INDEX = 0  # The only cached position; payload position value is -1.
DIRECTIONS: list[torch.Tensor | np.ndarray] = []
AGG_BY: list[str] | None = ['time_horizon_months']
PCA_MODEL_LOAD_PATH: Path | None = None
PCA_MODEL_SAVE_PATH: Path | None = None  # For example: DATA_DIR / 'pca_model.pkl'
PHRASING_FIELDS = ['template_metadata.prompt_framing', 'template_metadata.output_format']

## 3. Download activation batches

Activation batches below the hardcoded bucket path are cached locally and skipped on later runs unless `OVERWRITE` is true. Uncached files download concurrently using `DOWNLOAD_WORKERS`; when `SELECTED_BATCH_FILES` is set, only those remote files are transferred.

In [ ]:
def parse_gcs_uri(uri):
    if not uri.startswith('gs://'):
        raise ValueError(f'Expected a gs:// URI, got {uri!r}')
    bucket, separator, prefix = uri[5:].partition('/')
    if not bucket or not separator or not prefix.strip('/'):
        raise ValueError(f'GCS URI must contain a bucket and prefix: {uri!r}')
    return bucket, prefix.strip('/')


client = storage.Client(project=PROJECT_ID)
bucket_name, activation_prefix = parse_gcs_uri(ACTIVATIONS_GCS_URI)
blobs = sorted(
    (
        blob for blob in client.bucket(bucket_name).list_blobs(prefix=activation_prefix.rstrip('/') + '/')
        if Path(blob.name).name.startswith('activations_batch_') and blob.name.endswith('.pt')
    ),
    key=lambda blob: blob.name,
)
if not blobs:
    raise FileNotFoundError(f'No activation batches found below {ACTIVATIONS_GCS_URI}')

available_blobs_by_name = {Path(blob.name).name: blob for blob in blobs}
if SELECTED_BATCH_FILES is None:
    blobs_to_download = blobs
else:
    missing_files = sorted(set(SELECTED_BATCH_FILES) - set(available_blobs_by_name))
    if missing_files:
        raise FileNotFoundError(f'Selected batch files are unavailable in GCS: {missing_files}')
    blobs_to_download = [available_blobs_by_name[name] for name in SELECTED_BATCH_FILES]

batch_dir = DATA_DIR / 'batches'
batch_dir.mkdir(parents=True, exist_ok=True)

def download_batch(blob):
    destination = batch_dir / Path(blob.name).name
    if OVERWRITE or not destination.exists():
        blob.download_to_filename(str(destination))
    return destination


with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    downloaded_batch_paths = list(tqdm(
        executor.map(download_batch, blobs_to_download),
        total=len(blobs_to_download),
        desc='Downloading batches',
    ))

print(f'GCS source: {ACTIVATIONS_GCS_URI}')
print(f'Downloaded/local batches: {len(downloaded_batch_paths):,}')

### 3.1 Select local batch files

Set `SELECTED_BATCH_FILES` to basenames such as `activations_batch_00000.pt` to download and read only those batches, or leave it as `None` to download and read every batch.

In [ ]:
downloaded_by_name = {path.name: path for path in downloaded_batch_paths}
if SELECTED_BATCH_FILES is None:
    selected_batch_paths = downloaded_batch_paths
else:
    missing_files = sorted(set(SELECTED_BATCH_FILES) - set(downloaded_by_name))
    if missing_files:
        raise FileNotFoundError(f'Selected batch files were not downloaded: {missing_files}')
    selected_batch_paths = [downloaded_by_name[name] for name in SELECTED_BATCH_FILES]

if not selected_batch_paths:
    raise ValueError('No local batch files were selected.')
print(f'Selected {len(selected_batch_paths):,} of {len(downloaded_batch_paths):,} local batches.')
for path in selected_batch_paths[:10]:
    print(path)

## 4. Filter metadata and load activations

Each batch already contains aligned `sample_indices`, `prompts`, `prompt_metadata`, and activations. The cell validates that contract, selects matching rows, and concatenates only `layer_out/21` at its sole cached position.

In [ ]:
_MISSING = object()


def get_metadata_field(metadata, dotted_path):
    value = metadata
    for key in dotted_path.split('.'):
        if not isinstance(value, dict) or key not in value:
            return _MISSING
        value = value[key]
    return value


def metadata_value_matches(actual_value, expected_value):
    if actual_value is _MISSING:
        return False
    allowed_values = expected_value if isinstance(expected_value, list) else [expected_value]
    return any(actual_value == allowed_value for allowed_value in allowed_values)


def load_batch(path):
    return torch.load(path, map_location='cpu', weights_only=True, mmap=True)


feature_parts = []
loaded_indices = []
selected_metadata = {}
absolute_positions = []
cached_position = None
for path in tqdm(selected_batch_paths, desc='Filtering activation batches'):
    payload = load_batch(path)
    sample_indices = payload['sample_indices']
    prompts = payload['prompts']
    metadata_rows = payload['prompt_metadata']
    tensor = payload['activations'][LAYER_COMPONENT]
    if not (len(sample_indices) == len(prompts) == len(metadata_rows) == tensor.shape[0]):
        raise ValueError(f'Misaligned rows in {path}')
    positions = list(payload['positions'])
    if POSITION_INDEX >= len(positions):
        raise ValueError(f'{path} has only {len(positions)} cached positions.')
    position_value = positions[POSITION_INDEX]
    if cached_position is None:
        cached_position = position_value
    elif position_value != cached_position:
        raise ValueError(f'Inconsistent cached positions in {path}: {positions}')
    row_offsets = [
        offset for offset, metadata in enumerate(metadata_rows)
        if all(
            metadata_value_matches(get_metadata_field(metadata, field), expected)
            for field, expected in (METADATA_FILTERS or {}).items()
        )
    ]
    if MAX_SAMPLES is not None:
        remaining = MAX_SAMPLES - len(loaded_indices)
        row_offsets = row_offsets[:max(remaining, 0)]
    if row_offsets:
        rows = torch.as_tensor(row_offsets, dtype=torch.long)
        feature_parts.append(tensor[:, POSITION_INDEX, :].index_select(0, rows).to(torch.float32))
        selected_rows = [(int(sample_indices[offset]), metadata_rows[offset]) for offset in row_offsets]
        loaded_indices.extend(sample_index for sample_index, _metadata in selected_rows)
        selected_metadata.update(selected_rows)
        absolute_positions.extend([position_value] * len(row_offsets))
    del tensor, payload
    if MAX_SAMPLES is not None and len(loaded_indices) >= MAX_SAMPLES:
        break

if not feature_parts:
    raise ValueError('No activation rows matched the configured filters.')
activation_matrix = torch.cat(feature_parts, dim=0)

for direction_index, direction in enumerate(DIRECTIONS):
    direction_tensor = torch.as_tensor(
        direction, dtype=activation_matrix.dtype, device=activation_matrix.device
    ).flatten()
    if direction_tensor.numel() != activation_matrix.shape[1]:
        raise ValueError(
            f'Direction {direction_index} has {direction_tensor.numel()} elements; '
            f'expected {activation_matrix.shape[1]}.'
        )
    direction_norm = torch.linalg.vector_norm(direction_tensor)
    if not torch.isfinite(direction_tensor).all() or not torch.isfinite(direction_norm) or direction_norm <= 0:
        raise ValueError(f'Direction {direction_index} must contain finite values and have nonzero norm.')
    unit_direction = direction_tensor / direction_norm
    activation_matrix = activation_matrix - (
        activation_matrix @ unit_direction
    ).unsqueeze(1) * unit_direction

selected_layer = LAYER_COMPONENT
del feature_parts
gc.collect()
print('Activation matrix shape:', tuple(activation_matrix.shape))
print(f'Sequentially removed {len(DIRECTIONS):,} direction(s) before aggregation and PCA.')
print(f'Loaded {len(loaded_indices):,} samples from {LAYER_COMPONENT} at position {cached_position}.')
print('First sample indices:', loaded_indices[:10])

## 6. Prepare metadata and project with PCA

Before aggregation, the notebook sequentially removes every configured direction from each activation vector, then normalizes each duration to `time_horizon_months` so equivalent values such as 60 minutes and 1 hour share a group. `AGG_BY` averages activation vectors for rows sharing the configured fields; use `None` or an empty list to project individual activation rows. Unless `PCA_MODEL_LOAD_PATH` is set, the final three-component projection fits scikit-learn's randomized PCA solver with a fixed seed. A loaded model only performs `transform` on the prepared matrix.

In [ ]:
def flatten_scalar_metadata(metadata, prefix=''):
    flattened = {}
    for key, value in metadata.items():
        path = f'{prefix}.{key}' if prefix else key
        if isinstance(value, dict):
            flattened.update(flatten_scalar_metadata(value, path))
        elif not isinstance(value, (list, tuple, set)):
            flattened[path] = value
    return flattened


metadata_df = pd.DataFrame([flatten_scalar_metadata(selected_metadata[index]) for index in loaded_indices])
metadata_df.insert(0, 'sample_index', loaded_indices)
metadata_df.insert(1, 'absolute_token_position', absolute_positions)
unit_to_months = {
    'second': 1 / (30.4375 * 86400), 'minute': 1 / (30.4375 * 1440),
    'hour': 1 / (30.4375 * 24), 'day': 1 / 30.4375, 'week': 7 / 30.4375,
    'month': 1, 'year': 12, 'decade': 120, 'century': 1200, 'millennium': 12000,
}
unit_to_months.update({f'{unit}s': value for unit, value in list(unit_to_months.items())})
unit_to_months['centuries'] = 1200
unit_to_months['millennia'] = 12000
value_field = 'base_value' if 'base_value' in metadata_df else 'value'
unit_field = 'base_unit' if 'base_unit' in metadata_df else 'unit'
unknown_units = sorted(set(metadata_df[unit_field].astype(str).str.lower()) - set(unit_to_months))
if unknown_units:
    raise ValueError(f'Cannot convert time-horizon units to months: {unknown_units}')
metadata_df['time_horizon_months'] = [
    round(float(value) * unit_to_months[str(unit).lower()], 12)
    for value, unit in zip(metadata_df[value_field], metadata_df[unit_field])
]
available_phrasing_fields = [field for field in PHRASING_FIELDS if field in metadata_df]
aggregation_fields = list(AGG_BY or [])
missing_aggregation_fields = set(aggregation_fields) - set(metadata_df.columns)
if missing_aggregation_fields:
    raise ValueError(f'Missing AGG_BY fields: {sorted(missing_aggregation_fields)}')
if len(set(aggregation_fields)) != len(aggregation_fields):
    raise ValueError('AGG_BY must not contain duplicate fields.')

if aggregation_fields:
    vectors, rows = [], []
    groups = metadata_df.groupby(aggregation_fields, dropna=False, sort=False).indices
    for row_offsets in groups.values():
        row_offsets = list(row_offsets)
        vectors.append(activation_matrix.index_select(0, torch.as_tensor(row_offsets)).mean(dim=0))
        row = metadata_df.iloc[row_offsets[0]].copy()
        row['source_sample_count'] = len(row_offsets)
        for field in [value_field, unit_field, *available_phrasing_fields]:
            if metadata_df.iloc[row_offsets][field].nunique(dropna=True) > 1:
                row[field] = '<averaged>'
        rows.append(row)
    pca_matrix = torch.stack(vectors)
    analysis_metadata_df = pd.DataFrame(rows).reset_index(drop=True)
else:
    pca_matrix = activation_matrix
    analysis_metadata_df = metadata_df.copy()
    analysis_metadata_df['source_sample_count'] = 1

pca_input = pca_matrix.cpu().numpy()
if PCA_MODEL_LOAD_PATH is None:
    if len(pca_input) < 3:
        raise ValueError('At least three analysis rows are required to fit a 3D PCA projection.')
    pca = PCA(n_components=3, svd_solver='randomized', random_state=0)
    projections = pca.fit_transform(pca_input)
    print('Fitted PCA model on the prepared activation matrix.')
else:
    pca_model_load_path = Path(PCA_MODEL_LOAD_PATH)
    with pca_model_load_path.open('rb') as model_file:
        pca = pickle.load(model_file)
    if not isinstance(pca, PCA):
        raise TypeError(f'{pca_model_load_path} does not contain a scikit-learn PCA model.')
    projections = pca.transform(pca_input)
    print(f'Loaded PCA model from {pca_model_load_path}.')

if projections.shape[1] != 3:
    raise ValueError(f'PCA model must produce exactly 3 components, got {projections.shape[1]}.')
if PCA_MODEL_SAVE_PATH is not None:
    pca_model_save_path = Path(PCA_MODEL_SAVE_PATH)
    pca_model_save_path.parent.mkdir(parents=True, exist_ok=True)
    with pca_model_save_path.open('wb') as model_file:
        pickle.dump(pca, model_file)
    print(f'Saved PCA model to {pca_model_save_path}.')

explained_fraction = pca.explained_variance_ratio_
print('PCA input shape:', tuple(pca_matrix.shape))
print('Explained variance fractions:', explained_fraction)

## 7. Visualize the projection

The controls discover scalar prompt-metadata fields automatically. The 3D projection and paired 2D projections have independent color and filtering controls.

In [ ]:
df_projs = analysis_metadata_df.copy().reset_index(drop=True)
df_projs[['PC1', 'PC2', 'PC3']] = projections
df_projs['log10_time_horizon_months'] = np.log10(df_projs['time_horizon_months'])
metadata_fields = sorted(column for column in df_projs if column not in {'PC1', 'PC2', 'PC3', 'sample_index'})
color_fields = ['log10_time_horizon_months', *[field for field in metadata_fields if field != 'log10_time_horizon_months']]
print(f'Prepared {len(df_projs):,} projected points.')

In [ ]:
import ipywidgets as widgets
import plotly.express as px
import plotly.io as pio
from IPython.display import display
from pandas.api.types import is_bool_dtype, is_numeric_dtype

try:
    from google.colab import output as colab_output
except ImportError:
    in_colab = False
else:
    in_colab = True
    colab_output.enable_custom_widget_manager()
    pio.renderers.default = 'colab'

color_dropdown = widgets.Dropdown(options=color_fields, value='log10_time_horizon_months', description='Color by:', layout=widgets.Layout(width='500px'))
filter_dropdown = widgets.Dropdown(options=[('(no filter)', None), *[(field, field) for field in metadata_fields]], description='Filter by:', layout=widgets.Layout(width='500px'))
filter_values = widgets.SelectMultiple(description='Keep:', rows=6, layout=widgets.Layout(width='500px'))
plot_output = widgets.Output()

def update_filter_values():
    field = filter_dropdown.value
    values = [] if field is None else sorted(df_projs[field].dropna().unique().tolist(), key=str)
    filter_values.options = [(str(value), value) for value in values]
    filter_values.value = tuple(values)
    filter_values.disabled = field is None

def render_plot(change=None):
    filtered = df_projs
    if filter_dropdown.value is not None:
        filtered = filtered[filtered[filter_dropdown.value].isin(filter_values.value)]
    plot_output.clear_output(wait=True)
    with plot_output:
        if filtered.empty:
            print('No points match the selected filter values.')
            return
        color_field = color_dropdown.value
        plot_data = filtered.copy()
        numeric_color = is_numeric_dtype(plot_data[color_field]) and not is_bool_dtype(plot_data[color_field])
        if not numeric_color:
            plot_data[color_field] = plot_data[color_field].astype('string').fillna('<missing>')
        title = f'{selected_layer}, cached position {POSITION_INDEX} ({len(plot_data):,} points)'
        hover_fields = ['sample_index', 'time_horizon_months']
        fig_3d = px.scatter_3d(plot_data, x='PC1', y='PC2', z='PC3', color=color_field, color_continuous_scale='Viridis' if numeric_color else None, hover_data=hover_fields, title=title, opacity=0.7)
        fig_3d.update_traces(marker={'size': 4})

        if in_colab:
            fig_3d.show(renderer='colab')
        else:
            display(fig_3d)

def on_filter_change(change):
    update_filter_values()
    render_plot()

color_dropdown.observe(render_plot, names='value')
filter_dropdown.observe(on_filter_change, names='value')
filter_values.observe(render_plot, names='value')
update_filter_values()
display(widgets.VBox([color_dropdown, filter_dropdown, filter_values, plot_output]))
render_plot()

### Paired 2D projections

These controls independently configure the PC1–PC2 and PC1–PC3 views.

In [ ]:
color_dropdown_2d = widgets.Dropdown(options=color_fields, value='log10_time_horizon_months', description='2D color:', layout=widgets.Layout(width='500px'))
filter_dropdown_2d = widgets.Dropdown(options=[('(no filter)', None), *[(field, field) for field in metadata_fields]], description='2D filter:', layout=widgets.Layout(width='500px'))
filter_values_2d = widgets.SelectMultiple(description='2D keep:', rows=6, layout=widgets.Layout(width='500px'))
plot_output_2d = widgets.Output()

def update_filter_values_2d():
    field = filter_dropdown_2d.value
    values = [] if field is None else sorted(df_projs[field].dropna().unique().tolist(), key=str)
    filter_values_2d.options = [(str(value), value) for value in values]
    filter_values_2d.value = tuple(values)
    filter_values_2d.disabled = field is None

def render_plot_2d(change=None):
    filtered = df_projs
    if filter_dropdown_2d.value is not None:
        filtered = filtered[filtered[filter_dropdown_2d.value].isin(filter_values_2d.value)]
    plot_output_2d.clear_output(wait=True)
    with plot_output_2d:
        if filtered.empty:
            print('No points match the selected 2D filter values.')
            return
        color_field = color_dropdown_2d.value
        plot_data = filtered.copy()
        numeric_color = is_numeric_dtype(plot_data[color_field]) and not is_bool_dtype(plot_data[color_field])
        if not numeric_color:
            plot_data[color_field] = plot_data[color_field].astype('string').fillna('<missing>')
        projection_pairs = pd.concat([
            plot_data.assign(_projection_pair='PC1 vs PC2', _pc_x=plot_data['PC1'], _pc_y=plot_data['PC2']),
            plot_data.assign(_projection_pair='PC1 vs PC3', _pc_x=plot_data['PC1'], _pc_y=plot_data['PC3']),
        ], ignore_index=True)
        title = f'{selected_layer}, cached position {POSITION_INDEX} ({len(plot_data):,} points)'
        fig_2d = px.scatter(
            projection_pairs,
            x='_pc_x',
            y='_pc_y',
            color=color_field,
            facet_col='_projection_pair',
            category_orders={'_projection_pair': ['PC1 vs PC2', 'PC1 vs PC3']},
            color_continuous_scale='Viridis' if numeric_color else None,
            hover_data=['sample_index', 'time_horizon_months'],
            labels={'_pc_x': 'PC1', '_pc_y': 'Component value', '_projection_pair': ''},
            title=f'2D projections - {title}',
            opacity=0.7,
        )
        fig_2d.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split('=')[-1]))
        fig_2d.update_traces(marker={'size': 5})
        fig_2d.update_xaxes(title_text='PC1')
        fig_2d.update_yaxes(title_text='PC2', col=1)
        fig_2d.update_yaxes(title_text='PC3', col=2)
        if in_colab:
            fig_2d.show(renderer='colab')
        else:
            display(fig_2d)

def on_filter_change_2d(change):
    update_filter_values_2d()
    render_plot_2d()

color_dropdown_2d.observe(render_plot_2d, names='value')
filter_dropdown_2d.observe(on_filter_change_2d, names='value')
filter_values_2d.observe(render_plot_2d, names='value')
update_filter_values_2d()
display(widgets.VBox([color_dropdown_2d, filter_dropdown_2d, filter_values_2d, plot_output_2d]))
render_plot_2d()

In [ ]:
df_projs